# A7 — Corrected undercut threshold

Answers Thraves' comment on `u = d_MEDIUM × 20 ≈ 0.74 s`:
*"better to explain how this estimation is being made, also, why it appears only for medium compound?"*

**Why medium is wrong.** The optimal strategy is soft → medium. At the stop the leader takes a
fresh medium while the rival stays out on a **worn soft**. The gain comes from how far the rival's
soft has degraded, so the soft wear rate applies, not the medium one.

**What the old formula was missing.** A fresh medium is inherently slower than a fresh soft, so
part of the degradation gain is given straight back. The corrected threshold is

    u = d_SOFT × a_w − (b_MEDIUM − b_SOFT)

where `a_w` is the rival's tyre age at the stop and `b_c` is compound c's base lap time when new.

**Why the base gap needs care.** `parameters.json` gives SOFT 88.55 and MEDIUM 94.55, implying a
6 s gap. That is a fitting artifact: SOFT was fitted on 2 circuits, MEDIUM on 7, so each intercept
absorbs a different set of track paces. This notebook re-estimates the gap with **circuit fixed
effects**, which is the only way to compare compounds on equal footing.

Run from `src/`. Needs `../data/clean_laps.csv` — see the export cell at the bottom if you
have not created it yet.

## 1. Get the calibration laps

Runs in this order, stopping at the first that works:

1. `../data/clean_laps.csv` if it already exists
2. a cached `clean` dataframe if `pitstops.ipynb` is already in memory
3. pulls from FastF1 directly, using the same filters as `pitstops.ipynb`

Option 3 takes a few minutes the first time, then caches in `f1_cache/`. Either way the
cleaned set is written to `../data/clean_laps.csv` so this only ever happens once.

In [6]:
import os
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

os.makedirs('../data', exist_ok=True)
PATH = '../data/clean_laps.csv'

HIGH_SPEED = ['Silverstone', 'Spa', 'Monza', 'Suzuka', 'Jeddah',
              'Albert Park', 'Las Vegas', 'Red Bull Ring']

def clean_laps(df):
    """Same filters as pitstops.ipynb."""
    d = df.copy()
    d = d[d['LapTimeS'].notna()]
    d = d[d['IsAccurate'] == True]
    d = d[d['Pitted'] == 0]
    d = d[d['LapNumber'] > 1]
    d = d[d['TrackStatus'].apply(lambda ts: all(c not in str(ts) for c in ['4','5','6','7']))]
    d = d.groupby('Track', group_keys=False).apply(
        lambda g: g[g['LapTimeS'] <= 1.07 * g['LapTimeS'].median()])
    return d

def pull_from_fastf1(year=2025):
    import fastf1
    os.makedirs('f1_cache', exist_ok=True)
    fastf1.Cache.enable_cache('f1_cache')
    frames = []
    for track in HIGH_SPEED:
        print(f"  loading {track}...")
        s = fastf1.get_session(year, track, 'R')
        s.load(telemetry=False, weather=True)
        lp = s.laps[['Driver','LapNumber','Compound','TyreLife','Stint',
                     'LapTime','PitInTime','PitOutTime','TrackStatus','IsAccurate']].copy()
        lp['LapTimeS'] = lp['LapTime'].dt.total_seconds()
        lp['Track'] = track
        frames.append(lp)
    df = pd.concat(frames, ignore_index=True)
    df['Pitted'] = (df['PitInTime'].notna() | df['PitOutTime'].notna()).astype(int)
    df = df[~df['Compound'].isin(['INTERMEDIATE', 'WET'])]
    return clean_laps(df)

# --- resolve the data, in order of cheapness ---
if os.path.exists(PATH):
    raw = pd.read_csv(PATH)
    print(f"loaded {PATH}")
elif 'clean' in dir() and isinstance(globals().get('clean'), pd.DataFrame):
    raw = globals()['clean'].copy()
    raw.to_csv(PATH, index=False)
    print(f"used the in-memory `clean` dataframe; saved to {PATH}")
else:
    print("no cached data - pulling from FastF1 (a few minutes the first time)")
    raw = pull_from_fastf1()
    raw.to_csv(PATH, index=False)
    print(f"saved {len(raw)} clean laps to {PATH}")

laps = raw.rename(columns={'LapTimeS':'lap_time', 'TyreLife':'tyre_age',
                           'LapNumber':'lap_number', 'Compound':'compound',
                           'Track':'circuit', 'Driver':'driver', 'Stint':'stint'})
laps['compound'] = laps['compound'].str.upper()
laps = laps.dropna(subset=['lap_time','tyre_age','lap_number','compound'])
laps = laps[laps['compound'].isin(['SOFT','MEDIUM','HARD'])]

if {'circuit','driver','stint'} <= set(laps.columns):
    laps['stint_id'] = (laps['circuit'].astype(str) + '_' +
                        laps['driver'].astype(str)  + '_' +
                        laps['stint'].astype(str))

print(f"\n{len(laps)} clean laps")
print(laps.groupby('compound').agg(n=('lap_time','size'), circuits=('circuit','nunique')))

loaded ../data/clean_laps.csv

5562 clean laps
             n  circuits
compound                
HARD      2438         8
MEDIUM    2450         8
SOFT       674         5


## 2. Per-compound wear rates, with standard errors

Laps within a stint are correlated, so errors are clustered by stint where possible.
This also settles the HARD-vs-MEDIUM ordering, which the paper currently reports without comment.

In [7]:
TYRE = {}
for c in ['SOFT', 'MEDIUM', 'HARD']:
    sub = laps[laps['compound'] == c]
    if len(sub) < 30:
        print(f"{c}: too few laps"); continue

    # circuit fixed effects: identify wear WITHIN a circuit, not across circuits
    formula = 'lap_time ~ tyre_age + lap_number + C(circuit)'
    f = smf.ols(formula, data=sub)
    res = (f.fit(cov_type='cluster', cov_kwds={'groups': sub['stint_id']})
           if 'stint_id' in sub.columns else f.fit())

    ci = res.conf_int().loc['tyre_age']
    TYRE[c] = {'wear': res.params['tyre_age'], 'se': res.bse['tyre_age'],
               'ci': [ci.iloc[0], ci.iloc[1]], 'r2': res.rsquared, 'n': len(sub),
               'circuits': sub['circuit'].nunique(),
               'corr': sub[['tyre_age','lap_number']].corr().iloc[0,1]}
    t = TYRE[c]
    print(f"{c:7s} wear {t['wear']:+.4f}  SE {t['se']:.4f}  "
          f"95% CI [{t['ci'][0]:+.4f}, {t['ci'][1]:+.4f}]  "
          f"R2 {t['r2']:.3f}  n {t['n']}  circuits {t['circuits']}  "
          f"corr(age,lap) {t['corr']:.2f}")

if {'MEDIUM', 'HARD'} <= set(TYRE):
    a, b = TYRE['MEDIUM']['ci'], TYRE['HARD']['ci']
    overlap = not (a[1] < b[0] or b[1] < a[0])
    print(f"\nMEDIUM and HARD intervals overlap: {overlap}")

SOFT    wear +0.0686  SE 0.0133  95% CI [+0.0426, +0.0946]  R2 0.946  n 674  circuits 5  corr(age,lap) 0.18
MEDIUM  wear +0.0220  SE 0.0063  95% CI [+0.0097, +0.0343]  R2 0.977  n 2450  circuits 8  corr(age,lap) 0.35
HARD    wear +0.0457  SE 0.0073  95% CI [+0.0314, +0.0600]  R2 0.978  n 2438  circuits 8  corr(age,lap) 0.47

MEDIUM and HARD intervals overlap: True


## 3. Base pace gap, with circuit fixed effects

The compound dummy is the pace difference between fresh tyres **holding circuit constant**.
Without the fixed effects this number is mostly track pace, not compound pace.

In [8]:
BASE_GAP = None
if 'circuit' in laps.columns and laps['circuit'].nunique() > 1:
    fe = smf.ols('lap_time ~ C(compound, Treatment(reference="SOFT")) '
                 '+ tyre_age + lap_number + C(circuit)', data=laps).fit()
    key = [p for p in fe.params.index if 'compound' in p and 'MEDIUM' in p][0]
    BASE_GAP = fe.params[key]
    lo, hi = fe.conf_int().loc[key]
    print(f"fresh MEDIUM slower than fresh SOFT by {BASE_GAP:.3f}s "
          f"(95% CI [{lo:.3f}, {hi:.3f}])")
    print(f"model R2 = {fe.rsquared:.3f}, {laps['circuit'].nunique()} circuits")

    naive = laps[laps.compound=='MEDIUM'].lap_time.mean() - laps[laps.compound=='SOFT'].lap_time.mean()
    print(f"\n(raw difference without fixed effects: {naive:.3f}s "
          f"- inflated by circuit composition, do not use)")
else:
    print("Need a circuit column with more than one circuit to identify the gap.")

fresh MEDIUM slower than fresh SOFT by 0.030s (95% CI [-0.059, 0.120])
model R2 = 0.977, 8 circuits

(raw difference without fixed effects: 5.447s - inflated by circuit composition, do not use)


## 4. The corrected threshold

In [9]:
A_W = 22     # rival's tyre age at the model's optimal stop; set to (optimal pit lap - 1)

d_soft = TYRE['SOFT']['wear']
gross  = d_soft * A_W
u_net  = gross - BASE_GAP

print(f"rival tyre age at the stop, a_w    : {A_W} laps")
print(f"soft degradation over {A_W} laps      : {gross:.2f} s")
print(f"fresh-medium pace penalty          : {BASE_GAP:.2f} s")
print(f"corrected undercut threshold, u    : {u_net:.2f} s")
print(f"\n(paper currently reports 0.74 s from d_MEDIUM x 20)")

print("\n--- sensitivity to a_w ---")
for a in [20, 21, 22, 23, 24]:
    print(f"  a_w={a}: u = {d_soft*a - BASE_GAP:.2f} s")

rival tyre age at the stop, a_w    : 22 laps
soft degradation over 22 laps      : 1.51 s
fresh-medium pace penalty          : 0.03 s
corrected undercut threshold, u    : 1.48 s

(paper currently reports 0.74 s from d_MEDIUM x 20)

--- sensitivity to a_w ---
  a_w=20: u = 1.34 s
  a_w=21: u = 1.41 s
  a_w=22: u = 1.48 s
  a_w=23: u = 1.55 s
  a_w=24: u = 1.62 s


## 5. Paste into the paper

Fill the bracketed values from the output above.

In [10]:
print(f'''
RESULTS, undercut subsection - replaces the u = d_MED x a_w line:

  The undercut is the gain a leader obtains by pitting one lap before a rival who stays
  out. On that lap the leader runs a fresh tyre while the rival runs a worn one, so the
  per-lap gain is the difference between their lap times:

      u = d_SOFT * a_w - (b_MEDIUM - b_SOFT)

  where d_SOFT is the soft compound's degradation rate, a_w is the rival's tyre age at the
  moment of the stop, and b_c is the base lap time of compound c when new. The second term
  accounts for a fresh medium being inherently slower than a fresh soft. Because the model's
  optimal strategy runs soft then medium, the rival at the stop is on a worn soft rather
  than a worn medium, so the soft degradation rate applies. Compound base pace was estimated
  with circuit fixed effects, since the per-compound intercepts are otherwise confounded with
  the differing sets of circuits on which each compound was run. With a_w = {A_W} laps this
  gives u = {u_net:.2f} s. A leader should cover a rival trailing by less than u, since a
  closer rival would emerge ahead after the stop; beyond that gap, holding position wins.
  This is a one-lap approximation: real undercuts accumulate over several laps as the
  rival's tyre continues to degrade.

RESULTS, calibration subsection - add:

  Fitted wear rates were SOFT {TYRE['SOFT']['wear']:.4f} (95% CI [{TYRE['SOFT']['ci'][0]:.4f}, {TYRE['SOFT']['ci'][1]:.4f}]),
  MEDIUM {TYRE['MEDIUM']['wear']:.4f} [{TYRE['MEDIUM']['ci'][0]:.4f}, {TYRE['MEDIUM']['ci'][1]:.4f}],
  and HARD {TYRE['HARD']['wear']:.4f} [{TYRE['HARD']['ci'][0]:.4f}, {TYRE['HARD']['ci'][1]:.4f}],
  with R-squared of {TYRE['SOFT']['r2']:.3f}, {TYRE['MEDIUM']['r2']:.3f}, and {TYRE['HARD']['r2']:.3f}.
  [If the medium and hard intervals overlap, add:] The medium and hard intervals overlap, so
  the numerically faster degradation of the hard compound is not distinguishable from sampling
  variation and likely reflects the conditions under which hard tyres are run - longer stints
  and different circuits - rather than the compound itself.

LIMITATIONS - add:

  The undercut calculation omits the out-lap penalty: a car leaving the pit lane is not at
  full pace on its first flying lap, which reduces the realised gain below the modelled value.
''')


RESULTS, undercut subsection - replaces the u = d_MED x a_w line:

  The undercut is the gain a leader obtains by pitting one lap before a rival who stays
  out. On that lap the leader runs a fresh tyre while the rival runs a worn one, so the
  per-lap gain is the difference between their lap times:

      u = d_SOFT * a_w - (b_MEDIUM - b_SOFT)

  where d_SOFT is the soft compound's degradation rate, a_w is the rival's tyre age at the
  moment of the stop, and b_c is the base lap time of compound c when new. The second term
  accounts for a fresh medium being inherently slower than a fresh soft. Because the model's
  optimal strategy runs soft then medium, the rival at the stop is on a worn soft rather
  than a worn medium, so the soft degradation rate applies. Compound base pace was estimated
  with circuit fixed effects, since the per-compound intercepts are otherwise confounded with
  the differing sets of circuits on which each compound was run. With a_w = 22 laps this
  gives u =